In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [1]:
!git clone https://github.com/amulya-714/GW-Glitch-Utility-Selection.git
%cd GW-Glitch-Utility-Selection

Cloning into 'GW-Glitch-Utility-Selection'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 73 (delta 6), reused 67 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 9.89 MiB | 14.72 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/GW-Glitch-Utility-Selection


In [ ]:
!ls

check_confidence.py	 inspect_dataset.py	src
check_labels.py		 inspect_mismatches.py	tests
check_shared_classes.py  pytest.ini		validate_labels.py
configs			 README.md
docs			 requirements.txt


In [ ]:
!cat requirements.txt

# Python 3.10-3.13. On Apple Silicon the default PyPI torch wheel includes MPS support.
torch>=2.5
numpy>=1.26
pandas>=2.2
pillow>=10.0
pyyaml>=6.0
scikit-learn>=1.4
matplotlib>=3.8
tqdm>=4.66
pytest>=8.0
requests


In [ ]:
!pip install -q "numpy>=1.26" "pandas>=2.2" "pillow>=10.0" "pyyaml>=6.0" "scikit-learn>=1.4" "matplotlib>=3.8" "tqdm>=4.66" "requests"

In [ ]:
import torch, numpy, pandas, sklearn
print("GPU:", torch.cuda.get_device_name(0))
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Sklearn:", sklearn.__version__)

GPU: Tesla T4
NumPy: 2.4.6
Pandas: 2.2.3
Sklearn: 1.6.1


In [ ]:
!mkdir -p data/raw
!wget -q -O data/raw/gravityspy.zip "https://zenodo.org/records/5649212/files/GravitySpy_O2_O3a.zip?download=1"

In [ ]:
!grep -n -A35 "metadata_dir\|metadata" configs/classifier.yaml

19:  metadata_dir: data/raw/metadata
20-  image_dir: data/raw/images          # raw PNG cache (never re-downloaded)
21-  processed_dir: data/processed       # preprocessed uint8 arrays (per split)
22-
23-data:
24-  ifo: H1                     # H1 or L1: separate experiments (no detector ID is a model input)
25-  view_columns: [url1, url2, url3, url4]
26-  view_durations: [0.5, 1.0, 2.0, 4.0]    # seconds; ORDER MUST MATCH view_columns
27-  # ---- preprocessing (see docs/preprocessing.md) -------------------------------------------------
28-  # APPROVED INITIAL BASELINE (chosen from the real-image review of H1 O2 contact sheets).
29-  # Pipeline: crop -> alpha composite -> resize (RGB) -> normalise. All four views are kept.
30-  # Originals are 800x600 RGBA PNGs and are never modified.
31-  #
32-  # Output size: the 573x479 crop has aspect 1.196. 128 x 153 (HxW) preserves it to 0.08% (integer rounding).
33-  # A square size would stretch it by ~16%, and is rejected by crop.max_aspect_d

In [ ]:
!unzip -o data/raw/gravityspy.zip -d data/raw/

Archive:  data/raw/gravityspy.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of data/raw/gravityspy.zip or
        data/raw/gravityspy.zip.zip, and cannot find data/raw/gravityspy.zip.ZIP, period.


In [ ]:
!find data/raw -maxdepth 3 -type f | head -40

data/raw/gravityspy.zip


In [ ]:
!unzip -l data/raw/gravityspy.zip | head -40

  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
Archive:  data/raw/gravityspy.zip
unzip:  cannot find zipfile directory in one of data/raw/gravityspy.zip or
        data/raw/gravityspy.zip.zip, and cannot find data/raw/gravityspy.zip.ZIP, period.


In [ ]:
!file data/raw/gravityspy.zip
!ls -lh data/raw/gravityspy.zip
!head -c 300 data/raw/gravityspy.zip

data/raw/gravityspy.zip: empty
-rw-r--r-- 1 root root 0 Nov 12 14:38 data/raw/gravityspy.zip


In [ ]:
!rm -f data/raw/gravityspy.zip

In [ ]:
from pathlib import Path
import requests

base = "https://zenodo.org/records/5649212/files/"
out = Path("data/raw/metadata")
out.mkdir(parents=True, exist_ok=True)

for name in ["H1_O2.csv", "H1_O3a.csv", "L1_O2.csv", "L1_O3a.csv"]:
    r = requests.get(base + name, params={"download": 1}, timeout=180)
    r.raise_for_status()
    (out / name).write_bytes(r.content)
    print(f"Downloaded {name}: {len(r.content)/1e6:.1f} MB")

Downloaded H1_O2.csv: 73.1 MB
Downloaded H1_O3a.csv: 90.2 MB
Downloaded L1_O2.csv: 65.8 MB
Downloaded L1_O3a.csv: 142.3 MB


In [ ]:
!git remote -v
!git log -1 --oneline
!git branch -a

origin	https://github.com/amulya-714/GW-Glitch-Utility-Selection.git (fetch)
origin	https://github.com/amulya-714/GW-Glitch-Utility-Selection.git (push)
5ae115a (HEAD -> main, origin/main, origin/HEAD) Update data and experiment ignore rules
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main


In [ ]:
!git ls-tree -r --name-only HEAD | grep '^src/'

src/__init__.py
src/evaluation/__init__.py
src/models/__init__.py
src/models/baseline_cnn.py
src/models/evaluate.py
src/models/losses.py
src/models/multiview_cnn.py
src/models/train.py
src/selection/__init__.py
src/utils.py


In [ ]:
!git pull origin main

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 12 (delta 3), reused 12 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 31.94 KiB | 6.39 MiB/s, done.
From https://github.com/amulya-714/GW-Glitch-Utility-Selection
 * branch            main       -> FETCH_HEAD
   5ae115a..2107c86  main       -> origin/main
Updating 5ae115a..2107c86
Fast-forward
 .gitignore                            |   2 +-
 src/data/__init__.py                  |   0
 src/data/crop_preview.py              | 481 ++++++++++++++++++++++++++++++++++
 src/data/dataset.py                   | 330 +++++++++++++++++++++++
 src/data/fake_data.py                 | 135 ++++++++++
 src/data/inspect_images.py            | 182 +++++++++++++
 src/data/preprocessing.py             | 363 +++++++++++++++++++++++++
 src/data/resolve_dataset_questions.py |  51 ++++
 src/data/splits.py                    | 367 +++++++++++++++

In [ ]:
!python -m pytest -q

........................................................................ [ 32%]
........................................................................ [ 64%]
........................................................................ [ 97%]
......                                                                   [100%]
222 passed in 258.97s (0:04:18)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p data/processed/arrays

!cp /content/drive/MyDrive/GW_Colab_Transfer/* data/processed/arrays/

In [ ]:
!ls -lh data/processed/arrays/H1_*

-rw------- 1 root root 321K Nov 12 15:21 data/processed/arrays/H1_train_b3ee1fe79d42.meta.json
-rw------- 1 root root 4.0G Nov 12 15:23 data/processed/arrays/H1_train_b3ee1fe79d42.npy
-rw------- 1 root root 155K Nov 12 15:23 data/processed/arrays/H1_val_680b42bea766.meta.json
-rw------- 1 root root 2.0G Nov 12 15:24 data/processed/arrays/H1_val_680b42bea766.npy


In [ ]:
import json, glob, os

for f in glob.glob("data/processed/arrays/H1_*.meta.json"):
    m = json.load(open(f))
    print(os.path.basename(f), len(m["kept_ids"]), m["pre"])

H1_val_680b42bea766.meta.json 8785 {'size': [128, 153], 'channels': 3, 'resize': 'bilinear', 'alpha': 'white', 'crop': [101, 61, 674, 540], 'crop_ref': [800, 600]}
H1_train_b3ee1fe79d42.meta.json 18233 {'size': [128, 153], 'channels': 3, 'resize': 'bilinear', 'alpha': 'white', 'crop': [101, 61, 674, 540], 'crop_ref': [800, 600]}


In [ ]:
!grep -n -A25 -B5 "def array_cache_key" src/data/dataset.py

146-
147-
148-# --------------------------------------------------------------------------- #
149-# Array cache
150-# --------------------------------------------------------------------------- #
151:def array_cache_key(ids: Sequence[str], pre_cfg: PreprocessConfig, view_columns: Sequence[str]) -> str:
152-    """Content key of a cached array: the ordered event IDs, EVERY geometry setting (crop box and reference size,
153-    channels, size, resize filter, alpha background) and the ORDERED view columns (which view fills which slot).
154-    Normalisation is applied on the fly to the uint8 array, so it is deliberately not part of the key."""
155-    return stable_hash({"ids": list(ids), "pre": pre_cfg.key(), "views": list(view_columns)})
156-
157-
158-def build_split_array(
159-    df: pd.DataFrame, store: ImageStore, pre_cfg: PreprocessConfig, view_columns: Sequence[str],
160-    processed_dir: str | Path, name: str, chunk: int = 1024, workers: int = 4,
161-) -> tuple[Path, pd.DataFram

In [ ]:
from src.models.train import load_config

In [ ]:
!sed -n '1,120p' src/models/train.py

"""Train a glitch classifier on SOURCE-run (O2) data only.

    python -m src.models.train --config configs/classifier.yaml --ifo H1 --model multiview
    python -m src.models.train --config configs/classifier.yaml --ifo H1 --debug        # tiny/fast

Protocol guarantees (see also splits.py):
  * only the source run is ever loaded (target runs raise TargetLeakageError);
  * train/val are a temporal split with a purge gap;
  * model selection (best epoch, early stopping) uses source-val macro-F1 only;
  * no detector or run identifier is a model input.
"""
from __future__ import annotations

import argparse
import logging
import math
import shutil
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from torch import nn

from src.data.dataset import ImageStore, class_counts, make_loader, make_sampler, prepare_split
from src.data.preprocessing import Augment, Normalizer, PreprocessConfig, preprocessing_signature
from

In [ ]:
from src.utils import load_config
from src.data.splits import build_source_splits
from src.data.preprocessing import PreprocessConfig
from src.data.dataset import array_cache_key
from pathlib import Path

cfg = load_config("configs/classifier.yaml")
cfg["data"]["ifo"] = "H1"

train_df, val_df, mapping, reports = build_source_splits(cfg, ifo="H1")
pre = PreprocessConfig.from_cfg(cfg["data"])
views = cfg["data"]["view_columns"]

for name, df, old_key in [
    ("train", train_df, "b3ee1fe79d42"),
    ("val", val_df, "680b42bea766")
]:
    key = array_cache_key(df.gravityspy_id.tolist(), pre, views)
    print(f"\n{name.upper()}")
    print("Current events:", len(df))
    print("Expected key:", key)
    print("Old key:", old_key)
    print("MATCH:", key.startswith(old_key))


TRAIN
Current events: 18267
Expected key: 1be3d090a4e9
Old key: b3ee1fe79d42
MATCH: False

VAL
Current events: 8785
Expected key: 680b42bea766
Old key: 680b42bea766
MATCH: True


In [ ]:
import json, shutil
from pathlib import Path

arr = Path("data/processed/arrays")
old = "H1_train_b3ee1fe79d42"
new = "H1_train_1be3d090a4e9"

meta = json.load(open(arr / f"{old}.meta.json"))
cached_ids = set(meta["kept_ids"])
current_ids = set(train_df["gravityspy_id"].astype(str))

print("Cached IDs:", len(cached_ids))
print("Missing from current metadata:", len(cached_ids - current_ids))

assert cached_ids.issubset(current_ids), "IDs don't match. Stop."

shutil.copy2(arr / f"{old}.npy", arr / f"{new}.npy")
shutil.copy2(arr / f"{old}.meta.json", arr / f"{new}.meta.json")

print("Training cache successfully linked!")

Cached IDs: 18233
Missing from current metadata: 0
Training cache successfully linked!


In [ ]:
from pathlib import Path
import json
from src.data.dataset import array_cache_key
from src.data.preprocessing import PreprocessConfig
from src.utils import resolve_path

# Verify both exact cache keys first
cfg = __import__("src.utils", fromlist=["load_config"]).load_config(
    "configs/classifier.yaml"
)
pre = PreprocessConfig.from_cfg(cfg["data"])
arr_dir = Path("data/processed/arrays")

for name, df in [("H1_train", train_df), ("H1_val", val_df)]:
    key = array_cache_key(df.gravityspy_id.tolist(), pre, cfg["data"]["view_columns"])
    print(name, key, (arr_dir / f"{name}_{key}.npy").exists())

H1_train 1be3d090a4e9 True
H1_val 680b42bea766 True


In [ ]:
from pathlib import Path

p = Path("src/data/dataset.py")
s = p.read_text()

old = '''    dl_fail = store.ensure_downloaded(df, d["view_columns"], desc=f"download[{name}]")
    if len(dl_fail):
        bad = set(dl_fail["gravityspy_id"])
        df = df.loc[~df["gravityspy_id"].isin(bad)].reset_index(drop=True)
    path, kept, dec_fail = build_split_array(
        df, store, pre, d["view_columns"], resolve_path(cfg["paths"]["processed_dir"]), name)
    failures = pd.concat([dl_fail, dec_fail], ignore_index=True)
'''

new = '''    processed_dir = resolve_path(cfg["paths"]["processed_dir"])
    key = array_cache_key(df["gravityspy_id"].tolist(), pre, d["view_columns"])
    cache_dir = Path(processed_dir) / "arrays"
    arr_path = cache_dir / f"{name}_{key}.npy"
    meta_path = cache_dir / f"{name}_{key}.meta.json"

    if arr_path.exists() and meta_path.exists():
        path, kept, dec_fail = build_split_array(
            df, store, pre, d["view_columns"], processed_dir, name)
        dl_fail = pd.DataFrame(columns=["gravityspy_id", "view", "url", "reason"])
    else:
        dl_fail = store.ensure_downloaded(df, d["view_columns"], desc=f"download[{name}]")
        if len(dl_fail):
            bad = set(dl_fail["gravityspy_id"])
            df = df.loc[~df["gravityspy_id"].isin(bad)].reset_index(drop=True)
        path, kept, dec_fail = build_split_array(
            df, store, pre, d["view_columns"], processed_dir, name)
    failures = pd.concat([dl_fail, dec_fail], ignore_index=True)
'''

assert old in s, "Expected source block not found. No changes made."
p.write_text(s.replace(old, new))
print("Patched successfully.")

Patched successfully.


In [ ]:
!python -m src.models.train \
  --config configs/classifier.yaml \
  --ifo H1 \
  --model multiview \
  --name H1_O2_colab_baseline_v4 \
  --device cuda \
  --set train.batch_size=32 \
  --set train.num_workers=2

15:35:05 | INFO    | train | Run: H1_O2_colab_baseline_v4 | device: cuda | seed: 42
15:35:05 | INFO    | train | Preprocessing [dbdbffab0c2b]: crop=[101, 61, 674, 540] ref=[800, 600] size(HxW)=[128, 153] channels=3 resize=bilinear alpha=white norm=unit views=['url1', 'url2', 'url3', 'url4']
15:35:06 | INFO    | src.data.splits | H1_O2: 70871 -> 70871 events after cleaning
15:35:06 | INFO    | src.data.splits | H1_O2: 782 events with label 'None_of_the_Above' are outside the trained label space (excluded by config)
15:35:06 | INFO    | src.data.splits | split[chronological]: train=55917 val=14018 purged=154 (gap=21600s)
15:35:06 | WARNING | src.data.splits | val class 'Chirp' has only 3 event(s) (< 20): its metrics will be unreliable
15:35:06 | INFO    | src.data.splits | TRAIN class counts (before/after confidence filter and cap):
                     split  conf>=0.7  final(cap)
ml_label                                         
1080Lines            31642      30248        3000
Blip   

In [ ]:
!ls -lh experiments/runs/H1_O2_colab_baseline_v4/checkpoints/

total 12M
-rw-r--r-- 1 root root 5.7M Nov 12 17:16 best.pt
-rw-r--r-- 1 root root 5.7M Nov 12 17:32 last.pt


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/metadata/H1_O3a.csv")

known_classes = [
    "1080Lines", "1400Ripples", "Air_Compressor", "Blip",
    "Chirp", "Extremely_Loud", "Helix", "Koi_Fish",
    "Light_Modulation", "Low_Frequency_Burst",
    "Low_Frequency_Lines", "No_Glitch", "Paired_Doves",
    "Power_Line", "Repeating_Blips", "Scattered_Light",
    "Scratchy", "Tomte", "Violin_Mode", "Wandering_Line",
    "Whistle"
]

df = df[df["ml_label"].isin(known_classes)]
sample = (
    df.groupby("ml_label", group_keys=False)
      .apply(lambda x: x.sample(min(len(x), 100), random_state=42))
      .reset_index(drop=True)
)

sample.to_csv("data/raw/metadata/H1_O3a_sample.csv", index=False)

print("Sample size:", len(sample))
print(sample["ml_label"].value_counts())

Sample size: 1892
ml_label
1400Ripples            100
Blip                   100
Air_Compressor         100
Koi_Fish               100
Extremely_Loud         100
Tomte                  100
Repeating_Blips        100
Light_Modulation       100
Low_Frequency_Burst    100
No_Glitch              100
Low_Frequency_Lines    100
Paired_Doves           100
Power_Line             100
Whistle                100
Violin_Mode            100
Scattered_Light        100
Scratchy               100
Helix                   98
Wandering_Line          39
Chirp                   32
1080Lines               23
Name: count, dtype: int64


/tmp/ipykernel_380/3108798559.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 100), random_state=42))


In [ ]:
!python -m src.models.evaluate --help

usage: evaluate.py [-h] --run-dir RUN_DIR [--split {val,target}]
                   [--checkpoint CHECKPOINT] [--final] [--allow-rerun]
                   [--device DEVICE]

Evaluation: metrics, source-validation evaluation, and the gated final target-run evaluation.

CLI
---
Source-run validation (safe, repeatable):
    python -m src.models.evaluate --run-dir experiments/runs/<run> --split val

FINAL cross-run evaluation on the unseen target run (O3a). Run ONCE per frozen model:
    python -m src.models.evaluate --run-dir experiments/runs/<run> --split target --final

Target-run guard rails:
  * `--final` is mandatory for --split target.
  * The checkpoint is the one chosen on source-val only (best.pt); nothing is tuned here.
  * Thresholds come from config (evaluate.target_confidence_thresholds), all are reported.
  * A final_eval_log.json is written; a second target run is refused unless --allow-rerun
    is passed, so repeated peeking is visible in the record.

options:
  -h, --hel

In [ ]:
!grep -n -E "H1_O3a|metadata|read_csv|prepare_split|ensure_downloaded" src/models/evaluate.py

33:from src.data.dataset import ImageStore, make_loader, prepare_split
35:    ClassMapping, apply_class_mapping, build_source_splits, cap_per_class, clean_metadata,
36:    load_run_metadata,
202:    ds, kept, _ = prepare_split(val_df, f"{cfg['data']['ifo']}_val", cfg, store)
334:        raw = load_run_metadata(resolve_path(cfg["paths"]["metadata_dir"]), ifo, run, d["view_columns"],
336:        clean = clean_metadata(raw, ifo, d["view_columns"], name)
350:        ds, kept, failures = prepare_split(full, f"{name}_target", cfg, store)


In [ ]:
!sed -n '310,365p' src/models/evaluate.py

            "n_events": n_total,
        },
        "definitions": REPORT_DEFINITIONS,
    }
    return report, closed


def evaluate_target_final(run_dir: Path, cfg: dict, model: nn.Module, mapping: ClassMapping,
                          device: torch.device, ckpt_path: Path, allow_rerun: bool = False) -> dict[str, Any]:
    """Evaluate on every protocol target run for the configured detector. Never trains or tunes."""
    log_path = run_dir / "final_eval_log.json"
    if log_path.exists() and not allow_rerun:
        raise RuntimeError(
            f"{log_path} exists: this model already had its final target evaluation. "
            "Re-evaluating the same model on the target is a form of peeking. "
            "Pass --allow-rerun only if you are sure (it will be recorded).")

    d, proto, ev = cfg["data"], cfg["protocol"], cfg["evaluate"]
    ifo, seed = d["ifo"], cfg["experiment"]["seed"]
    cap = resolve_target_cap(cfg)   # None = every eligible event
    store = ImageStore.fr

In [ ]:
from pathlib import Path
import shutil

original = Path("experiments/runs/H1_O2_colab_baseline_v4")
sample_run = Path("experiments/runs/H1_O2_colab_baseline_v4_sample_diagnostic")

if sample_run.exists():
    raise FileExistsError(f"{sample_run} already exists. Stop and tell me.")

shutil.copytree(original, sample_run)

p = Path("src/models/evaluate.py")
s = p.read_text()

old = '''        raw = load_run_metadata(resolve_path(cfg["paths"]["metadata_dir"]), ifo, run, d["view_columns"],
                                target_runs=proto["target_runs"], purpose="final_eval")'''

new = '''        if run_dir.name.endswith("_sample_diagnostic"):
            sample_path = resolve_path(cfg["paths"]["metadata_dir"]) / f"{name}_sample.csv"
            raw = pd.read_csv(sample_path)
            log.warning("%s: PRELIMINARY SAMPLED evaluation using %s", name, sample_path)
        else:
            raw = load_run_metadata(resolve_path(cfg["paths"]["metadata_dir"]), ifo, run, d["view_columns"],
                                    target_runs=proto["target_runs"], purpose="final_eval")'''

assert old in s, "Could not find the expected evaluator code. No patch applied."
p.write_text(s.replace(old, new, 1))

print("Separate diagnostic run created:", sample_run)
print("Evaluator patched for this diagnostic run only.")

Separate diagnostic run created: experiments/runs/H1_O2_colab_baseline_v4_sample_diagnostic
Evaluator patched for this diagnostic run only.


In [ ]:
!python -m src.models.evaluate \
  --run-dir experiments/runs/H1_O2_colab_baseline_v4_sample_diagnostic \
  --split target \
  --final \
  --device cuda

17:42:25 | INFO    | __main__ | Loaded experiments/runs/H1_O2_colab_baseline_v4_sample_diagnostic/checkpoints/best.pt on cuda
17:42:25 | WARNING | __main__ | H1_O3a: PRELIMINARY SAMPLED evaluation using /content/GW-Glitch-Utility-Selection/data/raw/metadata/H1_O3a_sample.csv
17:42:25 | INFO    | src.data.splits | H1_O3a: 1892 -> 1892 events after cleaning
17:42:25 | INFO    | __main__ | H1_O3a: evaluating ALL 1892 eligible events (7568 images to fetch/cache)
17:42:25 | INFO    | src.data.dataset | download[H1_O3a_target]: 7568 images needed, 4 cached, 7564 to download
download[H1_O3a_target]: 100% 7564/7564 [06:40<00:00, 18.91it/s]
17:49:06 | WARNING | src.data.dataset | download[H1_O3a_target]: 4 image(s) failed to download (their events will be dropped)
decode[H1_O3a_target]: 100% 2/2 [02:38<00:00, 79.24s/it]
17:51:44 | WARNING | __main__ | H1_O3a: 1 of 1892 target events (0.053%) could not be evaluated because an image view failed to download/decode. They are listed in target_H1_O3a